# Polaris Availability Test — Daily Watchdog

**Realm:** POLARIS (default — no separate realm created)  
**Bootstrap:** uses root token to create watchdog principal on first run (idempotent)

## Test sequence
```
T01: OAuth2 token acquisition     → auth + PostgreSQL
T02: Catalog list                 → API layer
T03: Namespace CRUD               → write path + entity_version
T04: Table CRUD                   → metadata + MinIO location
T05: RBAC enforcement (negative)  → authorization system
T06: Health correlation           → detect entity_version mismatch
```

In [14]:
import sys
sys.path.insert(0, '.')
import importlib
import polaris_test_utils
importlib.reload(polaris_test_utils)
from polaris_test_utils import *

tr = TestResult()

✅ Polaris test utils loaded
   Polaris: http://192.168.139.2:8181  | Realm: POLARIS
   OpenSearch: https://localhost:9200


In [13]:
# Cell 2: One-time bootstrap (idempotent — safe to re-run every day)
# First run: creates watchdog-principal under default POLARIS realm, prints clientSecret
# Later runs: detects existing resources, does nothing destructive

client_id, client_secret_from_setup = ensure_watchdog_setup()
client_id, new_secret = reset_watchdog_principal()
# ⚠️ FIRST RUN ONLY: copy the printed clientSecret below and keep it
# (e.g. paste into a K8s secret). On later runs, set it here manually
# instead of relying on client_secret_from_setup (which will be None).

WATCHDOG_CLIENT_ID = client_id or "99fc1f19d76f5ee7"
WATCHDOG_SECRET = new_secret or "4a50f0a84b9752c391fa2700d4425296"


🔧 Checking watchdog setup (realm: POLARIS)...
  ✅ principal exists: watchdog-principal
  ✅ principal role exists: watchdog-principal-role
  ✅ principal role assigned (500)
  ✅ catalog exists: watchdog-catalog
  ✅ catalog role exists: watchdog-catalog-role
  ✅ grants verified
🔧 Watchdog setup complete (realm: POLARIS)

🔄 Resetting watchdog principal...
  DELETE existing principal: 204
  ✅ principal recreated
     clientId:     99fc1f19d76f5ee7
     clientSecret: 4a50f0a84b9752c391fa2700d4425296
     ⚠️  Save this secret — Polaris will not show it again.
  ✅ principal role re-assigned (201)


In [15]:
# T01: OAuth2 token acquisition (watchdog principal, default realm)
print("T01: OAuth2 token acquisition")
try:
    token = get_watchdog_token(WATCHDOG_CLIENT_ID, WATCHDOG_SECRET)
    tr.record("T01_auth", True, "token acquired")
except AssertionError as e:
    tr.record("T01_auth", False, str(e))
    print("⛔ Cannot continue without token")
    raise

T01: OAuth2 token acquisition
  ✅ T01_auth: token acquired


In [16]:
# T02: Catalog list
print("\nT02: Catalog list")
r = requests.get(f"{BASE_MGMT}/catalogs", headers=h(token))
passed = r.status_code == 200
detail = f"HTTP {r.status_code}"
if passed:
    names = [c["name"] for c in r.json().get("catalogs", [])]
    detail = f"catalogs visible: {names}"
tr.record("T02_catalog_list", passed, detail)


T02: Catalog list
  ❌ T02_catalog_list: HTTP 403


In [ ]:
# T03: Namespace CRUD (within watchdog-catalog)
print("\nT03: Namespace CRUD")

r_create = create_namespace(catalog=WATCHDOG_CATALOG, ns=WATCHDOG_NAMESPACE, token=token)
passed_create = r_create.status_code in (200, 201)
tr.record("T03a_namespace_create", passed_create, f"HTTP {r_create.status_code}")

r_get = requests.get(
    f"{BASE_CAT}/{WATCHDOG_CATALOG}/namespaces/{WATCHDOG_NAMESPACE}",
    headers=h(token)
)
passed_get = r_get.status_code == 200
tr.record("T03b_namespace_read", passed_get, f"HTTP {r_get.status_code}")

tr.record("T03_namespace_crud", passed_create and passed_get,
    "create + read OK" if passed_create and passed_get else "FAILED")

In [ ]:
# T04: Table CRUD
print("\nT04: Table CRUD")

r_create = requests.post(
    f"{BASE_CAT}/{WATCHDOG_CATALOG}/namespaces/{WATCHDOG_NAMESPACE}/tables",
    headers=h(token),
    json={
        "name": WATCHDOG_TABLE,
        "location": f"s3a://data-catalog-bucket/{WATCHDOG_CATALOG}/{WATCHDOG_NAMESPACE}/{WATCHDOG_TABLE}/",
        "schema": {
            "type": "struct",
            "fields": [
                {"id": 1, "name": "id",    "type": "long",   "required": True},
                {"id": 2, "name": "value", "type": "string", "required": False}
            ]
        }
    }
)
passed_create = r_create.status_code in (200, 201)
tr.record("T04a_table_create", passed_create, f"HTTP {r_create.status_code}")

r_get = requests.get(
    f"{BASE_CAT}/{WATCHDOG_CATALOG}/namespaces/{WATCHDOG_NAMESPACE}/tables/{WATCHDOG_TABLE}",
    headers=h(token)
)
passed_get = r_get.status_code == 200
tr.record("T04b_table_read", passed_get, f"HTTP {r_get.status_code}")

tr.record("T04_table_crud", passed_create and passed_get,
    "create + read OK" if passed_create and passed_get else "FAILED")

In [ ]:
# T05: RBAC enforcement (negative test)
# watchdog-principal only has CATALOG_MANAGE_CONTENT on watchdog-catalog —
# it must NOT be able to delete an unrelated catalog (e.g. test-error-catalog
# if present, or any catalog it wasn't granted on).
print("\nT05: RBAC enforcement — watchdog must NOT manage other catalogs")

other_catalogs = [
    c["name"] for c in requests.get(f"{BASE_MGMT}/catalogs", headers=h(root_token())).json().get("catalogs", [])
    if c["name"] != WATCHDOG_CATALOG
]

if other_catalogs:
    target = other_catalogs[0]
    r = requests.delete(f"{BASE_MGMT}/catalogs/{target}", headers=h(token))
    passed = r.status_code in (403, 401)
    tr.record("T05_rbac_enforcement", passed,
        f"HTTP {r.status_code} on DELETE {target} ({'RBAC working' if passed else 'RBAC BROKEN'})")
else:
    tr.record("T05_rbac_enforcement", True, "skipped — no other catalog to test against")

In [ ]:
# T06: Health correlation — detect entity_version mismatch
print("\nT06: Health correlation")
r_health = requests.get(f"{POLARIS_URL}/q/health")
health_up = r_health.status_code == 200 and r_health.json().get("status") == "UP"
write_failed = not tr.results.get("T03_namespace_crud", {}).get("passed", True)

if write_failed and health_up:
    tr.record("T06_health_correlation", False,
        "⚠️  entity_version mismatch suspected — health=UP but write failed")
    print()
    print("⚠️  ENTITY VERSION MISMATCH DETECTED")
    print("   Fix (ask system engineer to run):")
    print("     kubectl rollout restart deployment/benchmarks-polaris -n datahub-hynix")
    print("   DO NOT restart PostgreSQL")
else:
    tr.record("T06_health_correlation", True,
        f"health={'UP' if health_up else 'DOWN'} write={'OK' if not write_failed else 'FAIL'}")

In [ ]:
# Cleanup — only watchdog's own table + namespace (not the whole catalog)
print("\nCleanup...")
requests.delete(
    f"{BASE_CAT}/{WATCHDOG_CATALOG}/namespaces/{WATCHDOG_NAMESPACE}/tables/{WATCHDOG_TABLE}",
    headers=h(token)
)
requests.delete(
    f"{BASE_CAT}/{WATCHDOG_CATALOG}/namespaces/{WATCHDOG_NAMESPACE}",
    headers=h(token)
)
print("  table + namespace deleted (catalog + principal kept for next run)")

In [ ]:
# Summary
print()
all_passed = tr.summary()
print()
if all_passed:
    print("✅ Polaris fully operational")
else:
    print("❌ Issues detected — check failed tests above")
    print()
    print("Common fixes:")
    print("  (this notebook has no kubectl access — report below to system engineer)")
    print("  T01 fail: ask to check pod status: kubectl get pods -n datahub-hynix | grep polaris")
    print("  T03 fail + T06 warn: ask to restart: kubectl rollout restart deployment/benchmarks-polaris")
    print("  T05 fail: CHECK RBAC IMMEDIATELY — privileges too broad")